<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.2-transformer-from-scratch/notebooks/exercises-2.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 2.2: Transformer From Scratch — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 2 | 6 exercises: attention matrix, multi-head attention, positional encoding, parameter counting, KV-cache, full mini-transformer.

---

In [ ]:
!pip install torch -q


---
## Exercise 1 (Easy): Attention Matrix Visualization
Compute Q·Kᵀ/√d attention scores for a toy example and visualize which tokens attend to which.

In [ ]:
import torch
import torch.nn.functional as F

# Toy example: 4 tokens, d_model=8
torch.manual_seed(42)
seq_len, d_model = 4, 8
x = torch.randn(1, seq_len, d_model)

# Q, K, V projections (simplified)
W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

Q = x @ W_q
K = x @ W_k
V = x @ W_v

# Scaled dot-product attention
scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)
print('Raw attention scores (Q·Kᵀ/√d):')
print(scores[0].round(decimals=2))

# Apply causal mask (GPT-style)
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
scores_masked = scores.masked_fill(mask, float('-inf'))

# Softmax
attn_weights = F.softmax(scores_masked, dim=-1)
print('\nAttention weights (after causal mask + softmax):')
print(attn_weights[0].round(decimals=2))
print('\nRow 1: token 1 only sees itself (1.00)')
print('Row 4: token 4 attends to all previous tokens')
print('This IS the causal mask from Lesson 2.4!')


---
## Exercise 2 (Easy): Multi-Head Attention
Split Q, K, V into multiple heads and show they learn different patterns.

In [ ]:
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = q @ k.transpose(-2, -1) / (self.d_k ** 0.5)
        mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out), attn

mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(1, 6, 64)  # 6 tokens
out, attn = mha(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Attention: {attn.shape} (batch, heads, seq, seq)')
print(f'\nEach head learns different patterns:')
for h in range(4):
    print(f'  Head {h}: max attention = {attn[0,h].max():.2f}, pattern diversity = {attn[0,h].std():.3f}')


---
## Exercise 3 (Medium): Positional Encoding
Compare sinusoidal vs learned positional encodings. Show that without them, order doesn't matter.

In [ ]:
import torch
import math

def sinusoidal_pe(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = sinusoidal_pe(20, 64)
print('Sinusoidal PE shape:', pe.shape)
print(f'Position 0: [{pe[0,:4].tolist()}...]')
print(f'Position 1: [{pe[1,:4].tolist()}...]')

# Show: nearby positions are similar, far positions are different
import torch.nn.functional as F
sim_01 = F.cosine_similarity(pe[0:1], pe[1:2]).item()
sim_010 = F.cosine_similarity(pe[0:1], pe[10:11]).item()
print(f'\nCosine similarity pos 0 vs 1:  {sim_01:.3f} (nearby = similar)')
print(f'Cosine similarity pos 0 vs 10: {sim_010:.3f} (far = different)')
print('\nModern models use RoPE instead (rotary positional encoding)')


---
## Exercise 4 (Medium): Parameter Counting
Count parameters in GPT-2 and understand where the 124M comes from.

In [ ]:
def count_params(vocab=50257, d=768, layers=12, heads=12, ctx=1024):
    tok_emb = vocab * d
    pos_emb = ctx * d
    per_layer = (
        4 * d * d +    # Q, K, V, O projections
        4 * d +        # biases
        2 * (d * 4*d) + # FFN (up + down)
        2 * 4*d +      # FFN biases
        2 * 2 * d      # LayerNorm (2 per layer)
    )
    total = tok_emb + pos_emb + layers * per_layer + d  # + final LN
    return {
        'Token embeddings': tok_emb,
        'Position embeddings': pos_emb,
        'Per layer': per_layer,
        f'{layers} layers total': layers * per_layer,
        'TOTAL': total,
    }

configs = [
    ('GPT-2 Small',  50257, 768,  12, 12, 1024),
    ('GPT-2 Medium', 50257, 1024, 24, 16, 1024),
    ('GPT-2 Large',  50257, 1280, 36, 20, 1024),
    ('GPT-2 XL',     50257, 1600, 48, 25, 1024),
]

print('PARAMETER COUNTING')
for name, *args in configs:
    params = count_params(*args)
    total = params['TOTAL']
    print(f'\n  {name}: {total:,} params ({total/1e6:.0f}M)')
    for k, v in params.items():
        if k != 'TOTAL':
            print(f'    {k}: {v:,} ({v/total*100:.1f}%)')


---
## Exercise 5 (Hard): KV-Cache Implementation
Implement KV-cache to speed up autoregressive generation.

In [ ]:
import torch
import torch.nn as nn

class CachedAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.cache_k = None
        self.cache_v = None
    
    def forward(self, x, use_cache=False):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        if use_cache and self.cache_k is not None:
            k = torch.cat([self.cache_k, k], dim=2)
            v = torch.cat([self.cache_v, v], dim=2)
        
        if use_cache:
            self.cache_k = k
            self.cache_v = v
        
        scores = q @ k.transpose(-2, -1) / (self.d_k ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        return (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)

# Demo
attn = CachedAttention()

# Prefill: process entire prompt at once
prompt = torch.randn(1, 5, 64)  # 5 tokens
out1 = attn(prompt, use_cache=True)
print(f'Prefill: processed {prompt.shape[1]} tokens, cache size: {attn.cache_k.shape[2]}')

# Decode: one token at a time, reusing cache
for i in range(3):
    new_token = torch.randn(1, 1, 64)
    out = attn(new_token, use_cache=True)
    print(f'Decode step {i+1}: 1 new token, cache size: {attn.cache_k.shape[2]}')

print('\nWithout cache: recompute ALL tokens at each step → O(n²)')
print('With cache: only compute NEW token\'s Q against cached K,V → O(n)')


---
## Exercise 6 (Challenge): Build a Mini Transformer
Combine multi-head attention, FFN, and layer norm into a complete transformer block.

In [ ]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, d_ff=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)  # Pre-norm (modern style)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),  # Modern activation
            nn.Linear(d_ff, d_model),
        )
    
    def forward(self, x):
        T = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(T)
        # Pre-norm + residual connection
        h = self.ln1(x)
        x = x + self.attn(h, h, h, attn_mask=mask, is_causal=True)[0]
        x = x + self.ffn(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size=1000, d_model=64, n_layers=2, n_heads=4):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        B, T = x.shape
        x = self.tok_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln_f(x))

model = MiniGPT()
total_params = sum(p.numel() for p in model.parameters())
x = torch.randint(0, 1000, (1, 10))
logits = model(x)
print(f'MiniGPT: {total_params:,} parameters')
print(f'Input:   {x.shape} (batch=1, seq=10)')
print(f'Output:  {logits.shape} (batch=1, seq=10, vocab=1000)')
print(f'\nThis IS GPT — just smaller! Same architecture as GPT-2/3/4.')
print(f'Scale up d_model, n_layers, n_heads, vocab → GPT-4')


---
**6 exercises complete.** Transformer concepts mastered: attention scores, multi-head attention, positional encoding, parameter counting, KV-cache, full transformer block.